In [1]:
from typing import Optional, Tuple

import numpy as np
import pandas as pd


KM_S_PER_AU_YR = 4.74047


def radec_to_unit_vector(
    ra_deg: np.ndarray,
    dec_deg: np.ndarray,
) -> np.ndarray:
    """
    Convierte RA, Dec en grados a vectores unitarios ICRS.
    """

    ra_rad = np.deg2rad(ra_deg)
    dec_rad = np.deg2rad(dec_deg)

    return np.column_stack(
        (
            np.cos(dec_rad) * np.cos(ra_rad),
            np.cos(dec_rad) * np.sin(ra_rad),
            np.sin(dec_rad),
        )
    )


def simulate_proper_motions_from_fixed_apex(
    df: pd.DataFrame,
    apex_ra_deg: float,
    apex_dec_deg: float,
    speed_kms: float,
    parallax_mas: Optional[float] = None,
    ra_col: str = "ra",
    dec_col: str = "dec",
    parallax_col: str = "parallax",

    pmra_log_error_range: Tuple[float, float] = (-2.0, 0.0),
    pmra_log_error_center: float = -1.0,

    pmdec_log_error_range: Tuple[float, float] = (-2.0, 0.0),
    pmdec_log_error_center: float = -1.0,

    parallax_log_error_range: Tuple[float, float] = (-2.0, 0.0),
    parallax_log_error_center: float = -1.0,

    apply_pmra_error: bool = True,
    apply_pmdec_error: bool = True,
    apply_parallax_error: bool = False,
    seed: Optional[int] = None,
    copy: bool = True,
) -> pd.DataFrame:
    """
    Simula movimientos propios Gaia para estrellas de un cúmulo con:

        - ápex fijo en RA/Dec;
        - velocidad espacial común;
        - paralaje verdadera fija opcional;
        - errores opcionales e independientes en pmra, pmdec y parallax.

    Gaia usa:

        pmra = mu_alpha* = mu_alpha cos(dec)
        pmdec = mu_delta

    Parameters
    ----------
    df : pd.DataFrame
        Catálogo de estrellas. Debe contener ra y dec.
        Si parallax_mas=None, debe contener parallax_col.

    apex_ra_deg : float
        Ascensión recta del ápex, en grados.

    apex_dec_deg : float
        Declinación del ápex, en grados.

    speed_kms : float
        Velocidad espacial común del cúmulo, en km/s.

    parallax_mas : float or None
        Paralaje verdadera fija del cúmulo, en mas.
        Si se pasa, todas las estrellas usan esta paralaje verdadera.
        Si es None, se toma la paralaje verdadera desde df[parallax_col].

    pmra_log_error_range : tuple
        Rango uniforme para el error de pmra:

            sigma_pmra = 10**Uniform(a, b)

        Ejemplo:
            (-2, 0) produce errores entre 0.01 y 1 mas/yr.

    pmdec_log_error_range : tuple
        Rango uniforme para el error de pmdec.

    parallax_log_error_range : tuple
        Rango uniforme para el error de paralaje:

            sigma_parallax = 10**Uniform(a, b)

        En mas.

    apply_pmra_error : bool
        Si True, perturba pmra.

    apply_pmdec_error : bool
        Si True, perturba pmdec.

    apply_parallax_error : bool
        Si True, perturba parallax.

    seed : int or None
        Semilla aleatoria.

    copy : bool
        Si True, no modifica el DataFrame original.

    Returns
    -------
    pd.DataFrame
        DataFrame con columnas nuevas:

            parallax_true
            parallax_error_mas
            parallax_noise_mas
            parallax
            distance_pc_true
            apex_ra_deg
            apex_dec_deg
            speed_kms
            lambda_deg
            sin_lambda
            cos_lambda
            vt_kms_true
            radial_velocity_kms_true
            mu_total_masyr_true
            pmra_true
            pmdec_true
            pmra_error_masyr
            pmdec_error_masyr
            pmra_noise_masyr
            pmdec_noise_masyr
            pmra
            pmdec
    """

    required = [ra_col, dec_col]

    if parallax_mas is None:
        required.append(parallax_col)

    missing = [col for col in required if col not in df.columns]

    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    rng = np.random.default_rng(seed)

    result = df.copy() if copy else df

    ra_deg = result[ra_col].to_numpy(dtype=float)
    dec_deg = result[dec_col].to_numpy(dtype=float)

    n_stars = len(result)

    # ------------------------------------------------------------
    # Paralaje verdadera
    # ------------------------------------------------------------

    if parallax_mas is not None:
        if parallax_mas <= 0 or not np.isfinite(parallax_mas):
            raise ValueError("parallax_mas must be positive and finite.")

        parallax_true = np.full(
            n_stars,
            float(parallax_mas),
            dtype=float,
        )
    else:
        parallax_true = result[parallax_col].to_numpy(dtype=float)

    valid_parallax_true = (
        np.isfinite(parallax_true)
        & (parallax_true > 0.0)
    )

    if not np.any(valid_parallax_true):
        raise ValueError("No valid positive true parallaxes were found.")

    distance_pc_true = np.full(n_stars, np.nan)
    distance_pc_true[valid_parallax_true] = (
        1000.0 / parallax_true[valid_parallax_true]
    )

    result["parallax_true"] = parallax_true
    result["distance_pc_true"] = distance_pc_true

    # ------------------------------------------------------------
    # Error de paralaje
    # Distribución triangular en espacio log10
    # ------------------------------------------------------------

    parallax_log_min, parallax_log_max = parallax_log_error_range

    if not (parallax_log_min <= parallax_log_error_center <= parallax_log_max):
        raise ValueError(
            "parallax_log_error_center debe estar entre "
            "parallax_log_error_range[0] y parallax_log_error_range[1]."
        )

    parallax_log_error = rng.triangular(
        left=parallax_log_min,
        mode=parallax_log_error_center,
        right=parallax_log_max,
        size=n_stars,
    )

    parallax_error_mas = 10.0 ** parallax_log_error

    if apply_parallax_error:
        parallax_noise_mas = rng.normal(
            loc=0.0,
            scale=parallax_error_mas,
            size=n_stars,
        )
    else:
        parallax_noise_mas = np.zeros(n_stars, dtype=float)

    parallax_observed = parallax_true + parallax_noise_mas

    result["parallax_log_error"] = parallax_log_error
    result["parallax_error_mas"] = parallax_error_mas
    result["parallax_noise_mas"] = parallax_noise_mas
    result[parallax_col] = parallax_observed

    # ------------------------------------------------------------
    # Vectores unitarios ICRS
    # ------------------------------------------------------------

    star_vec = radec_to_unit_vector(ra_deg, dec_deg)

    apex_vec = radec_to_unit_vector(
        np.array([apex_ra_deg], dtype=float),
        np.array([apex_dec_deg], dtype=float),
    )[0]

    apex_vec = apex_vec / np.linalg.norm(apex_vec)

    # ------------------------------------------------------------
    # Lambda: distancia angular estrella-ápex
    # ------------------------------------------------------------

    cos_lambda = np.clip(star_vec @ apex_vec, -1.0, 1.0)
    sin_lambda = np.sqrt(np.clip(1.0 - cos_lambda**2, 0.0, 1.0))
    lambda_rad = np.arctan2(sin_lambda, cos_lambda)

    result["apex_ra_deg"] = float(apex_ra_deg)
    result["apex_dec_deg"] = float(apex_dec_deg)
    result["speed_kms"] = float(speed_kms)

    result["cos_lambda"] = cos_lambda
    result["sin_lambda"] = sin_lambda
    result["lambda_deg"] = np.rad2deg(lambda_rad)

    # ------------------------------------------------------------
    # Velocidad tangencial verdadera
    #
    #     V_t = V sin(lambda)
    #
    # Movimiento propio total:
    #
    #     mu[mas/yr] = 1000 * V_t / (4.74047 * d[pc])
    #
    # IMPORTANTE:
    #     Para generar pmra_true y pmdec_true usamos la distancia verdadera,
    #     no la paralaje observada perturbada.
    # ------------------------------------------------------------

    vt_kms_true = speed_kms * sin_lambda

    mu_total_masyr_true = np.full(n_stars, np.nan)
    mu_total_masyr_true[valid_parallax_true] = (
        1000.0
        * vt_kms_true[valid_parallax_true]
        / (
            KM_S_PER_AU_YR
            * distance_pc_true[valid_parallax_true]
        )
    )

    result["vt_kms_true"] = vt_kms_true
    result["mu_total_masyr_true"] = mu_total_masyr_true

    result["radial_velocity_kms_true"] = speed_kms * cos_lambda

    # ------------------------------------------------------------
    # Dirección hacia el ápex proyectada sobre el plano tangente
    # ------------------------------------------------------------

    apex_tangent = apex_vec[None, :] - cos_lambda[:, None] * star_vec
    apex_tangent_norm = np.linalg.norm(apex_tangent, axis=1)

    tangent_hat = np.full_like(apex_tangent, np.nan)

    valid_tangent = (
        np.isfinite(apex_tangent_norm)
        & (apex_tangent_norm > 1e-15)
        & valid_parallax_true
    )

    tangent_hat[valid_tangent] = (
        apex_tangent[valid_tangent]
        / apex_tangent_norm[valid_tangent, None]
    )

    # ------------------------------------------------------------
    # Base tangente ecuatorial
    # ------------------------------------------------------------

    ra_rad = np.deg2rad(ra_deg)
    dec_rad = np.deg2rad(dec_deg)

    e_ra = np.column_stack(
        (
            -np.sin(ra_rad),
            np.cos(ra_rad),
            np.zeros_like(ra_rad),
        )
    )

    e_dec = np.column_stack(
        (
            -np.cos(ra_rad) * np.sin(dec_rad),
            -np.sin(ra_rad) * np.sin(dec_rad),
            np.cos(dec_rad),
        )
    )

    pmra_true = np.full(n_stars, np.nan)
    pmdec_true = np.full(n_stars, np.nan)

    pmra_true[valid_tangent] = (
        mu_total_masyr_true[valid_tangent]
        * np.sum(
            tangent_hat[valid_tangent]
            * e_ra[valid_tangent],
            axis=1,
        )
    )

    pmdec_true[valid_tangent] = (
        mu_total_masyr_true[valid_tangent]
        * np.sum(
            tangent_hat[valid_tangent]
            * e_dec[valid_tangent],
            axis=1,
        )
    )

    result["pmra_true"] = pmra_true
    result["pmdec_true"] = pmdec_true

    # ------------------------------------------------------------
    # Errores de movimientos propios
    # ------------------------------------------------------------

    # ------------------------------------------------------------
    # Errores de movimientos propios
    # Ahora se generan con distribución triangular en espacio log10
    # ------------------------------------------------------------

    pmra_log_min, pmra_log_max = pmra_log_error_range
    pmdec_log_min, pmdec_log_max = pmdec_log_error_range

    if not (pmra_log_min <= pmra_log_error_center <= pmra_log_max):
        raise ValueError(
            "pmra_log_error_center debe estar entre "
            "pmra_log_error_range[0] y pmra_log_error_range[1]."
        )

    if not (pmdec_log_min <= pmdec_log_error_center <= pmdec_log_max):
        raise ValueError(
            "pmdec_log_error_center debe estar entre "
            "pmdec_log_error_range[0] y pmdec_log_error_range[1]."
        )

    pmra_log_error = rng.triangular(
        left=pmra_log_min,
        mode=pmra_log_error_center,
        right=pmra_log_max,
        size=n_stars,
    )

    pmdec_log_error = rng.triangular(
        left=pmdec_log_min,
        mode=pmdec_log_error_center,
        right=pmdec_log_max,
        size=n_stars,
    )

    pmra_error_masyr = 10.0 ** pmra_log_error
    pmdec_error_masyr = 10.0 ** pmdec_log_error

    if apply_pmra_error:
        pmra_noise_masyr = rng.normal(
            loc=0.0,
            scale=pmra_error_masyr,
            size=n_stars,
        )
    else:
        pmra_noise_masyr = np.zeros(n_stars, dtype=float)

    if apply_pmdec_error:
        pmdec_noise_masyr = rng.normal(
            loc=0.0,
            scale=pmdec_error_masyr,
            size=n_stars,
        )
    else:
        pmdec_noise_masyr = np.zeros(n_stars, dtype=float)

    result["pmra_log_error"] = pmra_log_error
    result["pmdec_log_error"] = pmdec_log_error

    result["pmra_error_masyr"] = pmra_error_masyr
    result["pmdec_error_masyr"] = pmdec_error_masyr

    result["pmra_noise_masyr"] = pmra_noise_masyr
    result["pmdec_noise_masyr"] = pmdec_noise_masyr

    result["pmra"] = result["pmra_true"] + result["pmra_noise_masyr"]
    result["pmdec"] = result["pmdec_true"] + result["pmdec_noise_masyr"]

    result["pm_total_true_masyr"] = np.sqrt(
        result["pmra_true"]**2
        + result["pmdec_true"]**2
    )

    result["pm_total_observed_masyr"] = np.sqrt(
        result["pmra"]**2
        + result["pmdec"]**2
    )

    return result

In [2]:
df = pd.read_csv(r"C:\Users\nicob\One Drive Uniandes\OneDrive - Universidad de los Andes\Doctorado\proyecto\clusterization_project\dev\pasantia\analisis_detallado_hyades.csv")
df_sim = simulate_proper_motions_from_fixed_apex(
    df=df,
    apex_ra_deg=262.6,
    apex_dec_deg=-81.9,
    speed_kms=25.0,
    parallax_mas=np.sqrt(11.1**2+12.24**2+7.25**2),
    # pmra_log_error_range=(-2.0, 0.0),
    # pmra_log_error_center=-1.75,
    # pmdec_log_error_range=(-2.0, 0.0)
    # pmdec_log_error_center=-1.75,
    # parallax_log_error_range=(-3.0, -1.0),
    # parallax_log_error_center=-1.75,
    apply_pmra_error=False,
    apply_pmdec_error=False,
    apply_parallax_error=False,
    seed=42,
)
# df_sim = simulate_proper_motions_from_fixed_apex(
#     df=df_estrellas,
#     apex_ra_deg=120.0,
#     apex_dec_deg=35.0,
#     speed_kms=25.0,
#     pmra_log_error_range=(-2.0, 0.0),
#     pmdec_log_error_range=(-2.0, 0.0),
#     seed=42,
# )
df_sim.to_csv(r"C:\Users\nicob\One Drive Uniandes\OneDrive - Universidad de los Andes\Doctorado\proyecto\clusterization_project\data\datos_simulados\mock_open_cluster_gaia_hyades_clean.csv", index=False)

In [3]:
df = pd.read_csv(r"C:\Users\nicob\One Drive Uniandes\OneDrive - Universidad de los Andes\Doctorado\proyecto\clusterization_project\dev\pasantia\coma_sample.csv")
df_sim = simulate_proper_motions_from_fixed_apex(
    df=df,
    apex_ra_deg=170.54,
    apex_dec_deg=3.33,
    speed_kms=25.0,
    parallax_mas=11.6,
    # pmra_log_error_range=(-2.0, 0.0),
    # pmra_log_error_center=-1.75,
    # pmdec_log_error_range=(-2.0, 0.0)
    # pmdec_log_error_center=-1.75,
    # parallax_log_error_range=(-2.0, -1.0),
    # parallax_log_error_center=-1.75,
    apply_pmra_error=False,
    apply_pmdec_error=False,
    apply_parallax_error=False,
    seed=42,
)
# df_sim = simulate_proper_motions_from_fixed_apex(
#     df=df_estrellas,
#     apex_ra_deg=120.0,
#     apex_dec_deg=35.0,
#     speed_kms=25.0,
#     pmra_log_error_range=(-2.0, 0.0),
#     pmdec_log_error_range=(-2.0, 0.0),
#     seed=42,
# )
df_sim.to_csv(r"C:\Users\nicob\One Drive Uniandes\OneDrive - Universidad de los Andes\Doctorado\proyecto\clusterization_project\data\datos_simulados\mock_open_cluster_gaia_coma_clean.csv", index=False)

In [6]:
df = pd.read_csv(r"C:\Users\nicob\One Drive Uniandes\OneDrive - Universidad de los Andes\Doctorado\proyecto\clusterization_project\dev\pasantia\analisis_detallado_hyades.csv")
df_sim = simulate_proper_motions_from_fixed_apex(
    df=df,
    apex_ra_deg=95.24,
    apex_dec_deg=7.79,
    speed_kms=25.0,
    parallax_mas=20.7,
    pmra_log_error_range=(-2.0, 0.0),
    pmra_log_error_center=-1.75,
    # pmdec_log_error_range=(-2.0, 0.0)
    # pmdec_log_error_center=-1.75,
    # parallax_log_error_range=(-2.0, -1.0),
    # parallax_log_error_center=-1.75,
    apply_pmra_error=True,
    apply_pmdec_error=False,
    apply_parallax_error=False,
    seed=42,
)
# df_sim = simulate_proper_motions_from_fixed_apex(
#     df=df_estrellas,
#     apex_ra_deg=120.0,
#     apex_dec_deg=35.0,
#     speed_kms=25.0,
#     pmra_log_error_range=(-2.0, 0.0),
#     pmdec_log_error_range=(-2.0, 0.0),
#     seed=42,
# )
df_sim.to_csv(r"C:\Users\nicob\One Drive Uniandes\OneDrive - Universidad de los Andes\Doctorado\proyecto\clusterization_project\data\datos_simulados\mock_open_cluster_gaia_hyades_pmra.csv", index=False)

In [13]:
df = pd.read_csv(r"C:\Users\nicob\One Drive Uniandes\OneDrive - Universidad de los Andes\Doctorado\proyecto\clusterization_project\dev\pasantia\analisis_detallado_hyades.csv")
df_sim = simulate_proper_motions_from_fixed_apex(
    df=df,
    apex_ra_deg=95.24,
    apex_dec_deg=7.79,
    speed_kms=25.0,
    parallax_mas=20.7,
    # pmra_log_error_range=(-2.0, 0.0),
    # pmra_log_error_center=-1.75,
    pmdec_log_error_range=(-2.0, 0.0),
    pmdec_log_error_center=-1.75,
    # parallax_log_error_range=(-3.0, -1.0),
    # parallax_log_error_center=-1.75,
    apply_pmra_error=False,
    apply_pmdec_error=False,
    apply_parallax_error=False,
    seed=42,
)
# df_sim = simulate_proper_motions_from_fixed_apex(
#     df=df_estrellas,
#     apex_ra_deg=120.0,
#     apex_dec_deg=35.0,
#     speed_kms=25.0,
#     pmra_log_error_range=(-2.0, 0.0),
#     pmdec_log_error_range=(-2.0, 0.0),
#     seed=42,
# )
df_sim.to_csv(r"C:\Users\nicob\One Drive Uniandes\OneDrive - Universidad de los Andes\Doctorado\proyecto\clusterization_project\data\datos_simulados\mock_open_cluster_gaia_hyades_pmdec.csv", index=False)

In [12]:
df = pd.read_csv(r"C:\Users\nicob\One Drive Uniandes\OneDrive - Universidad de los Andes\Doctorado\proyecto\clusterization_project\dev\pasantia\analisis_detallado_hyades.csv")
df_sim = simulate_proper_motions_from_fixed_apex(
    df=df,
    apex_ra_deg=95.24,
    apex_dec_deg=7.79,
    speed_kms=25.0,
    parallax_mas=20.7,
    pmra_log_error_range=(-2.0, 0.0),
    pmra_log_error_center=-1.75,
    pmdec_log_error_range=(-2.0, 0.0),
    pmdec_log_error_center=-1.75,
    # parallax_log_error_range=(-3.0, -1.0),
    # parallax_log_error_center=-1.75,
    apply_pmra_error=False,
    apply_pmdec_error=False,
    apply_parallax_error=False,
    seed=42,
)
# df_sim = simulate_proper_motions_from_fixed_apex(
#     df=df_estrellas,
#     apex_ra_deg=120.0,
#     apex_dec_deg=35.0,
#     speed_kms=25.0,
#     pmra_log_error_range=(-2.0, 0.0),
#     pmdec_log_error_range=(-2.0, 0.0),
#     seed=42,
# )
df_sim.to_csv(r"C:\Users\nicob\One Drive Uniandes\OneDrive - Universidad de los Andes\Doctorado\proyecto\clusterization_project\data\datos_simulados\mock_open_cluster_gaia_hyades_pm.csv", index=False)

In [9]:
df = pd.read_csv(r"C:\Users\nicob\One Drive Uniandes\OneDrive - Universidad de los Andes\Doctorado\proyecto\clusterization_project\dev\pasantia\analisis_detallado_hyades.csv")
df_sim = simulate_proper_motions_from_fixed_apex(
    df=df,
    apex_ra_deg=95.24,
    apex_dec_deg=7.79,
    speed_kms=25.0,
    parallax_mas=20.7,
    # pmra_log_error_range=(-2.0, 0.0),
    # pmra_log_error_center=-1.75,
    # pmdec_log_error_range=(-2.0, 0.0)
    # pmdec_log_error_center=-1.75,
    parallax_log_error_range=(-3.0, -1.0),
    parallax_log_error_center=-1.75,
    apply_pmra_error=False,
    apply_pmdec_error=False,
    apply_parallax_error=False,
    seed=42,
)
# df_sim = simulate_proper_motions_from_fixed_apex(
#     df=df_estrellas,
#     apex_ra_deg=120.0,
#     apex_dec_deg=35.0,
#     speed_kms=25.0,
#     pmra_log_error_range=(-2.0, 0.0),
#     pmdec_log_error_range=(-2.0, 0.0),
#     seed=42,
# )
df_sim.to_csv(r"C:\Users\nicob\One Drive Uniandes\OneDrive - Universidad de los Andes\Doctorado\proyecto\clusterization_project\data\datos_simulados\mock_open_cluster_gaia_hyades_parallax.csv", index=False)

In [11]:
df = pd.read_csv(r"C:\Users\nicob\One Drive Uniandes\OneDrive - Universidad de los Andes\Doctorado\proyecto\clusterization_project\dev\pasantia\analisis_detallado_hyades.csv")
df_sim = simulate_proper_motions_from_fixed_apex(
    df=df,
    apex_ra_deg=95.24,
    apex_dec_deg=7.79,
    speed_kms=25.0,
    parallax_mas=20.7,
    pmra_log_error_range=(-2.0, 0.0),
    pmra_log_error_center=-1.75,
    pmdec_log_error_range=(-2.0, 0.0),
    pmdec_log_error_center=-1.75,
    parallax_log_error_range=(-3.0, -1.0),
    parallax_log_error_center=-1.75,
    apply_pmra_error=False,
    apply_pmdec_error=False,
    apply_parallax_error=False,
    seed=42,
)
# df_sim = simulate_proper_motions_from_fixed_apex(
#     df=df_estrellas,
#     apex_ra_deg=120.0,
#     apex_dec_deg=35.0,
#     speed_kms=25.0,
#     pmra_log_error_range=(-2.0, 0.0),
#     pmdec_log_error_range=(-2.0, 0.0),
#     seed=42,
# )
df_sim.to_csv(r"C:\Users\nicob\One Drive Uniandes\OneDrive - Universidad de los Andes\Doctorado\proyecto\clusterization_project\data\datos_simulados\mock_open_cluster_gaia_hyades_all.csv", index=False)

In [14]:
df = pd.read_csv(r"C:\Users\nicob\One Drive Uniandes\OneDrive - Universidad de los Andes\Doctorado\proyecto\clusterization_project\dev\pasantia\coma_sample.csv")
df_sim = simulate_proper_motions_from_fixed_apex(
    df=df,
    apex_ra_deg=170.54,
    apex_dec_deg=3.33,
    speed_kms=25.0,
    parallax_mas=11.6,
    pmra_log_error_range=(-2.0, 0.0),
    pmra_log_error_center=-1.75,
    # pmdec_log_error_range=(-2.0, 0.0)
    # pmdec_log_error_center=-1.75,
    # parallax_log_error_range=(-2.0, -1.0),
    # parallax_log_error_center=-1.75,
    apply_pmra_error=False,
    apply_pmdec_error=False,
    apply_parallax_error=False,
    seed=42,
)
# df_sim = simulate_proper_motions_from_fixed_apex(
#     df=df_estrellas,
#     apex_ra_deg=120.0,
#     apex_dec_deg=35.0,
#     speed_kms=25.0,
#     pmra_log_error_range=(-2.0, 0.0),
#     pmdec_log_error_range=(-2.0, 0.0),
#     seed=42,
# )
df_sim.to_csv(r"C:\Users\nicob\One Drive Uniandes\OneDrive - Universidad de los Andes\Doctorado\proyecto\clusterization_project\data\datos_simulados\mock_open_cluster_gaia_coma_pmra.csv", index=False)

In [16]:
df = pd.read_csv(r"C:\Users\nicob\One Drive Uniandes\OneDrive - Universidad de los Andes\Doctorado\proyecto\clusterization_project\dev\pasantia\coma_sample.csv")
df_sim = simulate_proper_motions_from_fixed_apex(
    df=df,
    apex_ra_deg=170.54,
    apex_dec_deg=3.33,
    speed_kms=25.0,
    parallax_mas=11.6,
    # pmra_log_error_range=(-2.0, 0.0),
    # pmra_log_error_center=-1.75,
    pmdec_log_error_range=(-2.0, 0.0),
    pmdec_log_error_center=-1.75,
    # parallax_log_error_range=(-2.0, -1.0),
    # parallax_log_error_center=-1.75,
    apply_pmra_error=False,
    apply_pmdec_error=False,
    apply_parallax_error=False,
    seed=42,
)
# df_sim = simulate_proper_motions_from_fixed_apex(
#     df=df_estrellas,
#     apex_ra_deg=120.0,
#     apex_dec_deg=35.0,
#     speed_kms=25.0,
#     pmra_log_error_range=(-2.0, 0.0),
#     pmdec_log_error_range=(-2.0, 0.0),
#     seed=42,
# )
df_sim.to_csv(r"C:\Users\nicob\One Drive Uniandes\OneDrive - Universidad de los Andes\Doctorado\proyecto\clusterization_project\data\datos_simulados\mock_open_cluster_gaia_coma_pmdec.csv", index=False)

In [17]:
df = pd.read_csv(r"C:\Users\nicob\One Drive Uniandes\OneDrive - Universidad de los Andes\Doctorado\proyecto\clusterization_project\dev\pasantia\coma_sample.csv")
df_sim = simulate_proper_motions_from_fixed_apex(
    df=df,
    apex_ra_deg=170.54,
    apex_dec_deg=3.33,
    speed_kms=25.0,
    parallax_mas=11.6,
    pmra_log_error_range=(-2.0, 0.0),
    pmra_log_error_center=-1.75,
    pmdec_log_error_range=(-2.0, 0.0),
    pmdec_log_error_center=-1.75,
    # parallax_log_error_range=(-2.0, -1.0),
    # parallax_log_error_center=-1.75,
    apply_pmra_error=False,
    apply_pmdec_error=False,
    apply_parallax_error=False,
    seed=42,
)
# df_sim = simulate_proper_motions_from_fixed_apex(
#     df=df_estrellas,
#     apex_ra_deg=120.0,
#     apex_dec_deg=35.0,
#     speed_kms=25.0,
#     pmra_log_error_range=(-2.0, 0.0),
#     pmdec_log_error_range=(-2.0, 0.0),
#     seed=42,
# )
df_sim.to_csv(r"C:\Users\nicob\One Drive Uniandes\OneDrive - Universidad de los Andes\Doctorado\proyecto\clusterization_project\data\datos_simulados\mock_open_cluster_gaia_coma_pm.csv", index=False)

In [18]:
df = pd.read_csv(r"C:\Users\nicob\One Drive Uniandes\OneDrive - Universidad de los Andes\Doctorado\proyecto\clusterization_project\dev\pasantia\coma_sample.csv")
df_sim = simulate_proper_motions_from_fixed_apex(
    df=df,
    apex_ra_deg=170.54,
    apex_dec_deg=3.33,
    speed_kms=25.0,
    parallax_mas=11.6,
    # pmra_log_error_range=(-2.0, 0.0),
    # pmra_log_error_center=-1.75,
    # pmdec_log_error_range=(-2.0, 0.0),
    # pmdec_log_error_center=-1.75,
    parallax_log_error_range=(-2.0, -1.0),
    parallax_log_error_center=-1.75,
    apply_pmra_error=False,
    apply_pmdec_error=False,
    apply_parallax_error=False,
    seed=42,
)
# df_sim = simulate_proper_motions_from_fixed_apex(
#     df=df_estrellas,
#     apex_ra_deg=120.0,
#     apex_dec_deg=35.0,
#     speed_kms=25.0,
#     pmra_log_error_range=(-2.0, 0.0),
#     pmdec_log_error_range=(-2.0, 0.0),
#     seed=42,
# )
df_sim.to_csv(r"C:\Users\nicob\One Drive Uniandes\OneDrive - Universidad de los Andes\Doctorado\proyecto\clusterization_project\data\datos_simulados\mock_open_cluster_gaia_coma_parallax.csv", index=False)

In [19]:
df = pd.read_csv(r"C:\Users\nicob\One Drive Uniandes\OneDrive - Universidad de los Andes\Doctorado\proyecto\clusterization_project\dev\pasantia\coma_sample.csv")
df_sim = simulate_proper_motions_from_fixed_apex(
    df=df,
    apex_ra_deg=170.54,
    apex_dec_deg=3.33,
    speed_kms=25.0,
    parallax_mas=11.6,
    pmra_log_error_range=(-2.0, 0.0),
    pmra_log_error_center=-1.75,
    pmdec_log_error_range=(-2.0, 0.0),
    pmdec_log_error_center=-1.75,
    parallax_log_error_range=(-2.0, -1.0),
    parallax_log_error_center=-1.75,
    apply_pmra_error=False,
    apply_pmdec_error=False,
    apply_parallax_error=False,
    seed=42,
)
# df_sim = simulate_proper_motions_from_fixed_apex(
#     df=df_estrellas,
#     apex_ra_deg=120.0,
#     apex_dec_deg=35.0,
#     speed_kms=25.0,
#     pmra_log_error_range=(-2.0, 0.0),
#     pmdec_log_error_range=(-2.0, 0.0),
#     seed=42,
# )
df_sim.to_csv(r"C:\Users\nicob\One Drive Uniandes\OneDrive - Universidad de los Andes\Doctorado\proyecto\clusterization_project\data\datos_simulados\mock_open_cluster_gaia_coma_all.csv", index=False)